In [1]:
from rich.console import Console
from dotenv import load_dotenv
import os
from openai import OpenAI
import json

load_dotenv(override=True)
openai = OpenAI()

console = Console()

todos = []
completed = []


def show(text):
    """
    Print text using Rich markup.
    """
    if text is None:
        return

    console.print(text)


def get_todo_report() -> str:
    """
    Return the full todo list.
    Completed todos are shown in green with strikethrough.
    """
    result = ""

    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"

    show(result)
    return result


def create_todos(descriptions: list[str]) -> str:
    """
    Add new todos.
    IMPORTANT: parameter name is descriptions because tool schema uses descriptions.
    """
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))

    return get_todo_report()


def mark_complete(index: int, completion_notes: str) -> str:
    """
    Mark one todo complete.
    index is 1-based.
    """
    if not 1 <= index <= len(todos):
        return "No todo at this index."

    completed[index - 1] = True

    show(f"[bold green]Completed Todo #{index}:[/bold green] {completion_notes}")

    return get_todo_report()


create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full todo list.",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                "type": "array",
                "items": {"type": "string"},
                "description": "A list of todo descriptions to add."
            }
        },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}


mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position, starting from 1, and return the full todo list.",
    "parameters": {
        "type": "object",
        "properties": {
            "index": {
                "type": "integer",
                "description": "The 1-based index of the todo to mark as complete."
            },
            "completion_notes": {
                "type": "string",
                "description": "Notes about how the todo was completed. Rich markup is allowed."
            }
        },
        "required": ["index", "completion_notes"],
        "additionalProperties": False
    }
}


tools = [
    {"type": "function", "function": create_todos_json},
    {"type": "function", "function": mark_complete_json}
]


available_tools = {
    "create_todos": create_todos,
    "mark_complete": mark_complete
}


def handle_tool_calls(tool_calls):
    """
    Run tool calls requested by the model.
    """
    results = []

    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        print(f"Handling tool call: {tool_name} with arguments: {arguments}", flush=True)

        tool = available_tools.get(tool_name)

        if tool is None:
            result = {
                "status": "error",
                "message": f"Tool {tool_name} not found."
            }
        else:
            result = tool(**arguments)

        results.append({
            "role": "tool",
            "content": json.dumps(result),
            "tool_call_id": tool_call.id
        })

    return results


def loop(messages):
    """
    Keep calling the model until it gives a final answer.
    If the model asks to use tools, run the tools first.
    """
    done = False

    while not done:
        response = openai.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools
        )

        finish_reason = response.choices[0].finish_reason

        if finish_reason == "tool_calls":
            assistant_message = response.choices[0].message
            tool_calls = assistant_message.tool_calls

            results = handle_tool_calls(tool_calls)

            messages.append(assistant_message)
            messages.extend(results)

        else:
            done = True

    final_answer = response.choices[0].message.content
    show(final_answer)

    return final_answer

In [2]:
todos, completed = [], []

create_todos(["Buy groceries", "Finish extra lab", "Eat banana"])

mark_complete(1, "Bought groceries from the store.")

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

Completed Todo #1: Bought groceries from the store.

Todo #1: Buy groceries
Todo #2: Finish extra lab
Todo #3: Eat banana

'Todo #1: [green][strike]Buy groceries[/strike][/green]\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [3]:
system_message = """
You are given a problem to solve by using your todo tools to plan a list of steps, then carrying out each step in turn.

You must:
1. Create a todo list for solving the problem.
2. Complete each todo one by one using the tools.
3. Reply with the final solution.

If any quantity is not provided in the question, include a step to make a reasonable estimate.

Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification.
Respond only with the answer after using your tools.
"""

user_message = """
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_message}
]

todos, completed = [], []

loop(messages)

Handling tool call: create_todos with arguments: {'descriptions': ['Define the distance between Boston and New York.', 'Calculate the time difference between the two trains leaving.', 'Estimate the distance from the speed and time it takes for both trains to meet.', 'Establish a formula to calculate the distance traveled for both trains until they meet.', 'Calculate the time when the two trains meet based on their speeds and the distance they traveled.']}


Todo #1: Define the distance between Boston and New York.
Todo #2: Calculate the time difference between the two trains leaving.
Todo #3: Estimate the distance from the speed and time it takes for both trains to meet.
Todo #4: Establish a formula to calculate the distance traveled for both trains until they meet.
Todo #5: Calculate the time when the two trains meet based on their speeds and the distance they traveled.

Handling tool call: mark_complete with arguments: {'index': 1, 'completion_notes': 'The distance between Boston and New York is estimated to be 215 miles.'}


Completed Todo #1: The distance between Boston and New York is estimated to be 215 miles.

Todo #1: Define the distance between Boston and New York.
Todo #2: Calculate the time difference between the two trains leaving.
Todo #3: Estimate the distance from the speed and time it takes for both trains to meet.
Todo #4: Establish a formula to calculate the distance traveled for both trains until they meet.
Todo #5: Calculate the time when the two trains meet based on their speeds and the distance they traveled.

...

Handling tool call: mark_complete with arguments: {'index': 5, 'completion_notes': 'Solving the equation 60t + 80(t - 1) = 215 gives t = 2.5 hours. Thus, the trains meet at 2:00 pm + 2.5 hours = 4:30 pm.'}


Completed Todo #5: Solving the equation 60t + 80(t - 1) = 215 gives t = 2.5 hours. Thus, the trains meet at 2:00 pm
+ 2.5 hours = 4:30 pm.

Todo #1: Define the distance between Boston and New York.
Todo #2: Calculate the time difference between the two trains leaving.
Todo #3: Estimate the distance from the speed and time it takes for both trains to meet.
Todo #4: Establish a formula to calculate the distance traveled for both trains until they meet.
Todo #5: Calculate the time when the two trains meet based on their speeds and the distance they traveled.

The two trains meet at 4:30 pm.

'The two trains meet at 4:30 pm.'

In [1]:
from rich.console import Console
from dotenv import load_dotenv
import os
from openai import OpenAI
import json
import gradio as gr
from pydantic import BaseModel
from pypdf import PdfReader
import requests
load_dotenv(override=True)
openai = OpenAI()

In [2]:
def show(text):
    try:
        console = Console()
    except Exception as e:
        print(f"Error initializing console: {e}")
        print(text)
        return

In [3]:
todos = []
completed = []

In [4]:
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"
    show(result)
    return result

def create_todos(description: list[str]) -> str: 
    todos.extend(description)
    completed.extend([False] * len(description))
    return get_todo_report()

def mark_complete(index:int, completion_notes: str) -> str: 
    if 1 <= index <= len(todos):
        completed[index-1] = True
    else:
        return "No todo at this index."
    
    Console().print(completion_notes)
    return get_todo_report()



get_todo_report()

''

In [5]:
todos, completed = [], []

create_todos(["Finish the report", "Call the client", "Schedule a meeting"])

mark_complete(2, "Called the client and discussed the project requirements.")

Called the client and discussed the project requirements.

'Todo #1: Finish the report\nTodo #2: [green][strike]Call the client[/strike][/green]\nTodo #3: Schedule a meeting\n'

In [6]:
todos, completed = [], []

create_todos(["Buy groceries", "Finish extra lab", "Eat banana"])

mark_complete(1, "bought")

bought

'Todo #1: [green][strike]Buy groceries[/strike][/green]\nTodo #2: Finish extra lab\nTodo #3: Eat banana\n'

In [7]:
create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}


mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [8]:
tools = [{"type": "function", "function": create_todos_json},
        {"type": "function", "function": mark_complete_json}]

In [9]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [10]:
def loop(messages):
    done = False
    while not done:
        response = openai.chat.completions.create(model="gpt-5.2", messages=messages, tools=tools, reasoning_effort="none")
        finish_reason = response.choices[0].finish_reason
        if finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)

In [11]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [12]:
todos, completed = [], []
loop(messages)

TypeError: create_todos() got an unexpected keyword argument 'descriptions'. Did you mean 'description'?